# Akurasi Training dan Testing: Random Forest (Dataset Zomato)

Notebook ini menghitung **akurasi training** dan **akurasi testing** model Random Forest untuk menebak kategori rating restoran (`Rating text`), lalu menampilkannya dalam chart.

**Cara pakai di Google Colab:**
1. Klik **Runtime → Run all**.
2. Saat diminta, **upload file CSV** dataset Zomato.
3. Tunggu sampai semua sel selesai (sekitar 1 menit).

| Istilah | Artinya |
|---|---|
| Akurasi training | Model menebak ulang restoran yang **sudah dipelajarinya** |
| Akurasi testing | Model menebak restoran yang **belum pernah dilihatnya** (kemampuan sebenarnya) |

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

warnings.filterwarnings("ignore")
KATEGORI = ["Poor", "Average", "Good", "Very Good", "Excellent"]

# ---- Gaya chart (palet aman untuk buta warna) ----
BIRU, ORANYE = "#2a78d6", "#eb6834"          # Training = biru, Testing = oranye
PERMUKAAN, TEKS, TEKS_2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e5e4e0"
RAMP_BIRU = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
plt.rcParams.update({
    "figure.facecolor": PERMUKAAN, "axes.facecolor": PERMUKAAN, "savefig.facecolor": PERMUKAAN,
    "axes.edgecolor": GRID, "axes.labelcolor": TEKS_2, "text.color": TEKS,
    "xtick.color": TEKS_2, "ytick.color": TEKS_2, "font.size": 11,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.axisbelow": True, "legend.frameon": False,
})


def angka(v, desimal=1):
    """Format angka gaya Indonesia: koma sebagai desimal."""
    return f"{v:.{desimal}f}".replace(".", ",")


def label_batang(ax, bars):
    """Tulis nilai di atas setiap batang (warna teks, bukan warna seri)."""
    for b in bars:
        ax.annotate(angka(b.get_height()) + "%", (b.get_x() + b.get_width() / 2, b.get_height()),
                    xytext=(0, 3), textcoords="offset points", ha="center", va="bottom",
                    fontsize=9, color=TEKS_2)

## 1. Upload dan baca dataset

In [ ]:
FILE_CSV = "Zomato Restaurant Dataset.csv"   # nama file; akan diganti otomatis setelah upload

import os
try:
    from google.colab import files           # hanya ada di Google Colab
    if not os.path.exists(FILE_CSV):
        print("Silakan upload file CSV dataset Zomato:")
        uploaded = files.upload()
        FILE_CSV = list(uploaded.keys())[0]
except ImportError:
    pass                                     # dijalankan di luar Colab: pakai FILE_CSV di atas

df = pd.read_csv(FILE_CSV, encoding="utf-8-sig")
print(f"Data awal: {len(df)} restoran, {df.shape[1]} kolom")

## 2. Bersihkan data

Restoran **"Not rated"** (rating 0, votes 0–3) dibuang karena tidak punya kunci jawaban. Sisanya adalah restoran yang punya rating.

In [ ]:
df = df[df["Rating text"] != "Not rated"].copy()
df["Cuisines"] = df["Cuisines"].fillna("")
print(f"Setelah buang 'Not rated': {len(df)} restoran")

jumlah = df["Rating text"].value_counts().reindex(KATEGORI)
fig, ax = plt.subplots(figsize=(8, 4.2))
bars = ax.bar(KATEGORI, jumlah.values, color=BIRU, width=0.6, edgecolor=PERMUKAAN, linewidth=1.5)
for b, n in zip(bars, jumlah.values):
    ax.annotate(f"{n:,}".replace(",", ".") + f" ({n / len(df):.1%})".replace(".", ","),
                (b.get_x() + b.get_width() / 2, n), xytext=(0, 3), textcoords="offset points",
                ha="center", va="bottom", fontsize=9, color=TEKS_2)
ax.set_title(f"Jumlah restoran per kategori rating ({len(df):,} restoran)".replace(",", "."))
ax.set_ylabel("Jumlah restoran")
ax.set_ylim(0, jumlah.max() * 1.15)
plt.tight_layout()
plt.show()

Kategori **tidak seimbang**: Average sangat dominan, sedangkan Poor dan Excellent sedikit. Karena itu, selain akurasi, perlu dilihat juga tebakan benar per kategori.

## 3. Siapkan fitur (X) dan target (y)

- **X (soal):** 28 fitur ciri restoran.
- **y (kunci jawaban):** `Rating text`.
- `Aggregate rating` dan `Rating color` **tidak dipakai**, karena itu bocoran jawaban.

In [ ]:
X = pd.DataFrame(index=df.index)
X["price_range"] = df["Price range"]
X["votes_log"] = np.log1p(df["Votes"])
X["table_booking"] = (df["Has Table booking"] == "Yes").astype(int)
X["online_delivery"] = (df["Has Online delivery"] == "Yes").astype(int)
X["india"] = (df["Country Code"] == 1).astype(int)
X["biaya_persentil"] = df.groupby("Currency")["Average Cost for two"].rank(pct=True)
X["jml_jenis_masakan"] = df["Cuisines"].str.count(",") + (df["Cuisines"] != "").astype(int)
X["jml_restoran_kota"] = df["City"].map(df["City"].value_counts())
for m in df["Cuisines"].str.split(",").explode().str.strip().value_counts().head(20).index:
    X["masakan_" + m] = df["Cuisines"].str.contains(m, regex=False).astype(int)

y = df["Rating text"]
print("Jumlah fitur:", X.shape[1])

## 4. Bagi data: 80% training, 20% testing

Kedua bagian **sama-sama punya rating**. Bedanya, rating data testing disembunyikan dari model saat menebak.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"Training: {len(X_train)} restoran")
print(f"Testing : {len(X_test)} restoran")

## 5. Latih Random Forest (200 pohon)

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)      # belajar dari ciri (X_train) DAN jawabannya (y_train)
print("Training selesai.")

## 6. Hitung akurasi training dan testing

Caranya sama untuk keduanya: **tebakan dicocokkan dengan rating asli, lalu dihitung benar / total**.

In [ ]:
tebakan_train = rf.predict(X_train)    # menebak ulang data yang sudah dipelajari
tebakan_test = rf.predict(X_test)      # menebak data yang belum pernah dilihat

benar_train = (tebakan_train == y_train.values).sum()
benar_test = (tebakan_test == y_test.values).sum()

print(f"Akurasi TRAINING = {benar_train} / {len(y_train)} = {benar_train / len(y_train):.1%}")
print(f"Akurasi TESTING  = {benar_test} / {len(y_test)} = {benar_test / len(y_test):.1%}")
print(f"Selisih          = {(benar_train / len(y_train) - benar_test / len(y_test)) * 100:.1f} poin")


def per_kategori(y_asli, tebakan):
    y_asli = np.asarray(y_asli)
    return pd.DataFrame({
        "Jumlah": [int((y_asli == k).sum()) for k in KATEGORI],
        "Benar": [int(((y_asli == k) & (tebakan == k)).sum()) for k in KATEGORI],
    }, index=KATEGORI)


tab_train = per_kategori(y_train, tebakan_train)
tab_test = per_kategori(y_test, tebakan_test)
tabel = pd.DataFrame({
    "Training: benar / jumlah": tab_train["Benar"].astype(str) + " / " + tab_train["Jumlah"].astype(str),
    "Testing: benar / jumlah": tab_test["Benar"].astype(str) + " / " + tab_test["Jumlah"].astype(str),
})
display(tabel)

In [ ]:
persen_train = tab_train["Benar"] / tab_train["Jumlah"] * 100
persen_test = tab_test["Benar"] / tab_test["Jumlah"] * 100

x = np.arange(len(KATEGORI)); w = 0.38
fig, ax = plt.subplots(figsize=(9, 4.5))
b1 = ax.bar(x - w / 2, persen_train, w, label="Training", color=BIRU, edgecolor=PERMUKAAN, linewidth=1.5)
b2 = ax.bar(x + w / 2, persen_test, w, label="Testing", color=ORANYE, edgecolor=PERMUKAAN, linewidth=1.5)
label_batang(ax, b1); label_batang(ax, b2)
ax.set_xticks(x, KATEGORI)
ax.set_ylim(0, 112)
ax.set_ylabel("Tebakan benar (%)")
ax.set_title("Random Forest: persen tebakan benar per kategori")
ax.legend(loc="upper center", ncol=2, bbox_to_anchor=(0.5, -0.1))
plt.tight_layout()
plt.show()

Di data training hampir semua kategori benar (model sudah hafal). Di data testing hasilnya turun, terutama untuk kategori yang datanya sedikit (Poor dan Excellent).

## 7. Confusion matrix (data testing)

Baris = rating asli, kolom = tebakan model. Angka di diagonal adalah tebakan yang benar.

In [ ]:
cm = confusion_matrix(y_test, tebakan_test, labels=KATEGORI)
cmap = LinearSegmentedColormap.from_list("biru", RAMP_BIRU)

fig, ax = plt.subplots(figsize=(7, 5.6))
im = ax.imshow(cm, cmap=cmap)
ax.grid(False)
for s in ax.spines.values():
    s.set_visible(False)
ax.set_xticks(range(5), KATEGORI)
ax.set_yticks(range(5), KATEGORI)
ax.set_xlabel("Tebakan model")
ax.set_ylabel("Rating asli")
ax.tick_params(length=0)
batas = cm.max() * 0.55
for i in range(5):
    for j in range(5):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=11,
                color="#ffffff" if cm[i, j] > batas else TEKS,
                fontweight="bold" if i == j else "normal")
ax.set_title("Confusion matrix Random Forest (data testing)")
cb = fig.colorbar(im, ax=ax, shrink=0.8)
cb.outline.set_visible(False)
cb.set_label("Jumlah restoran", color=TEKS_2)
plt.tight_layout()
plt.show()

## 8. Bandingkan 3 algoritma

Ketiga algoritma diuji dengan data, pembagian, dan fitur yang **sama persis**.

In [ ]:
algoritma = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": rf,
}
baris = []
for nama, model in algoritma.items():
    if nama != "Random Forest":
        model.fit(X_train, y_train)
    p_tr, p_te = model.predict(X_train), model.predict(X_test)
    baris.append({"Algoritma": nama,
                  "Akurasi training (%)": round(accuracy_score(y_train, p_tr) * 100, 1),
                  "Akurasi testing (%)": round(accuracy_score(y_test, p_te) * 100, 1)})
banding = pd.DataFrame(baris).set_index("Algoritma")
banding["Selisih (poin)"] = (banding["Akurasi training (%)"] - banding["Akurasi testing (%)"]).round(1)
display(banding)

x = np.arange(len(banding)); w = 0.38
fig, ax = plt.subplots(figsize=(8.5, 4.5))
b1 = ax.bar(x - w / 2, banding["Akurasi training (%)"], w, label="Training", color=BIRU, edgecolor=PERMUKAAN, linewidth=1.5)
b2 = ax.bar(x + w / 2, banding["Akurasi testing (%)"], w, label="Testing", color=ORANYE, edgecolor=PERMUKAAN, linewidth=1.5)
label_batang(ax, b1); label_batang(ax, b2)
ax.set_xticks(x, banding.index)
ax.set_ylim(0, 112)
ax.set_ylabel("Akurasi (%)")
ax.set_title("Akurasi training vs testing per algoritma")
ax.legend(loc="upper center", ncol=2, bbox_to_anchor=(0.5, -0.1))
plt.tight_layout()
plt.show()

- **Decision Tree:** hafal data training (99,7%), tapi testing-nya paling rendah. Ini overfitting paling parah.
- **Random Forest:** juga hafal data training, tapi testing-nya jauh lebih baik daripada Decision Tree.
- **Logistic Regression:** selisihnya kecil karena tidak menghafal, tapi kurang mampu menebak kategori yang datanya sedikit.

## 9. Fitur yang paling berpengaruh (Random Forest)

In [ ]:
nama_fitur = {"votes_log": "Votes", "biaya_persentil": "Biaya", "jml_restoran_kota": "Jumlah restoran di kota",
              "jml_jenis_masakan": "Jumlah jenis masakan", "price_range": "Price range",
              "online_delivery": "Online delivery", "india": "Negara (India)", "table_booking": "Table booking"}
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
imp.index = [nama_fitur.get(f, f.replace("masakan_", "Masakan: ")) for f in imp.index]
imp = imp[::-1]

fig, ax = plt.subplots(figsize=(8, 4.8))
bars = ax.barh(imp.index, imp.values, color=BIRU, height=0.6, edgecolor=PERMUKAAN, linewidth=1.5)
for b in bars:
    ax.annotate(angka(b.get_width(), 3), (b.get_width(), b.get_y() + b.get_height() / 2),
                xytext=(4, 0), textcoords="offset points", va="center", fontsize=9, color=TEKS_2)
ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
ax.spines["bottom"].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.set_xlim(0, imp.max() * 1.15)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: angka(v, 2)))
ax.set_xlabel("Importance")
ax.set_title("10 fitur paling berpengaruh")
plt.tight_layout()
plt.show()

## Ringkasan

1. Data training dan testing **sama-sama punya rating**. Restoran "Not rated" tidak dipakai.
2. **Akurasi training** = tebakan benar pada data yang sudah dipelajari, dibagi jumlah data training.
3. **Akurasi testing** = tebakan benar pada data yang belum pernah dilihat, dibagi jumlah data testing.
4. Yang dilaporkan sebagai kemampuan model adalah **akurasi testing**. Akurasi training dipakai untuk mendeteksi overfitting.